In [9]:
import shutil
import os

# Example: If your dataset is read-only in input, copy it over to working
if not os.path.exists('/kaggle/working/dataset/'):
    print("Moving to WORKING")
    shutil.copytree('/kaggle/input/datasets/alltimerookie/synthics-data-hackforhumanity', '/kaggle/working/dataset/')
    print("Moved to WORKING")

Moving to WORKING
Moved to WORKING


In [10]:
!pip install sentence-transformers scikit-learn xgboost pandas joblib

In [11]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import joblib
from sklearn.calibration import CalibratedClassifierCV

In [12]:
# 1. Load your data (Using dummy data for this example so you can run it immediately)
df = pd.read_csv("/kaggle/working/dataset/combined_data.csv")
print(df["status"].value_counts())
df = df.dropna(subset=["statement", "status"])
print(df["status"].value_counts())

status
Depression              12200
Normal                  12195
Suicidal                10851
Stress                   5200
Anxiety                  5200
Bipolar                  5200
Personality disorder     4200
Name: count, dtype: int64
status
Depression              12200
Normal                  12195
Suicidal                10851
Stress                   5200
Anxiety                  5200
Bipolar                  5200
Personality disorder     4200
Name: count, dtype: int64


In [13]:
# -------------------------------------------------------
# 2. Encode labels
# -------------------------------------------------------
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["status"])
print(df["label"].nunique())

joblib.dump(label_encoder, "label_encoder.pkl")

7


['label_encoder.pkl']

In [14]:
# -------------------------------------------------------
# 3. Train/validation/test split
# -------------------------------------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 44036
Val: 5505
Test: 5505


In [15]:
# -------------------------------------------------------
# 4. Generate embeddings
# -------------------------------------------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_text = train_df["statement"].tolist()
X_val_text = val_df["statement"].tolist()
X_test_text = test_df["statement"].tolist()

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

X_train = embedder.encode(X_train_text, show_progress_bar=True)
X_val = embedder.encode(X_val_text, show_progress_bar=True)
X_test = embedder.encode(X_test_text, show_progress_bar=True)

print("Train embedding shape:", X_train.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1377 [00:00<?, ?it/s]

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

Train embedding shape: (44036, 384)


In [16]:
# -------------------------------------------------------
# 5. Handle class imbalance
# -------------------------------------------------------
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [17]:
# -------------------------------------------------------
# 6. Train XGBoost
# -------------------------------------------------------
clf = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=50,
    device = "cuda"
)

clf.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50
)


[0]	validation_0-mlogloss:1.89382
[50]	validation_0-mlogloss:1.02345
[100]	validation_0-mlogloss:0.81228
[150]	validation_0-mlogloss:0.71223
[200]	validation_0-mlogloss:0.65189
[250]	validation_0-mlogloss:0.61069
[300]	validation_0-mlogloss:0.57981
[350]	validation_0-mlogloss:0.55622
[400]	validation_0-mlogloss:0.53655
[450]	validation_0-mlogloss:0.52047
[500]	validation_0-mlogloss:0.50749
[550]	validation_0-mlogloss:0.49654
[600]	validation_0-mlogloss:0.48721
[650]	validation_0-mlogloss:0.47883
[700]	validation_0-mlogloss:0.47194
[750]	validation_0-mlogloss:0.46592
[800]	validation_0-mlogloss:0.46135
[850]	validation_0-mlogloss:0.45700
[900]	validation_0-mlogloss:0.45365
[950]	validation_0-mlogloss:0.45029
[999]	validation_0-mlogloss:0.44780


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=50,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
# 2. Calibrate the model using the VALIDATION set (Never use the training set for this!)
# 'prefit' means we are calibrating an already trained model.
from sklearn.frozen import FrozenEstimator

# ---------------------------------------------------------
# 1. FIX XGBoost Device Mismatch
# ---------------------------------------------------------
# Tell XGBoost to use the CPU for calibration and inference.
# This stops the warning and is MUCH faster for small text batches.
clf.set_params(device='cpu')

# ---------------------------------------------------------
# 2. FIX Scikit-Learn Deprecation
# ---------------------------------------------------------
# Wrap your trained model in FrozenEstimator instead of using cv='prefit'
calibrated_clf = CalibratedClassifierCV(
    estimator=FrozenEstimator(clf), 
    method='isotonic'  # You can also use method='sigmoid'
)

calibrated_clf.fit(X_val, y_val)

CalibratedClassifierCV(estimator=FrozenEstimator(estimator=XGBClassifier(base_score=None,
                                                                         booster=None,
                                                                         callbacks=None,
                                                                         colsample_bylevel=None,
                                                                         colsample_bynode=None,
                                                                         colsample_bytree=0.8,
                                                                         device='cpu',
                                                                         early_stopping_rounds=50,
                                                                         enable_categorical=False,
                                                                         eval_metric='mlogloss',
                                                                         feature_types=None,
                                                                         feature_weights=None,
                                                                         gamma=None,
                                                                         grow_policy=None,
                                                                         importance_type=None,
                                                                         interaction_constraints=None,
                                                                         learning_rate=0.05,
                                                                         max_bin=None,
                                                                         max_cat_threshold=None,
                                                                         max_cat_to_onehot=None,
                                                                         max_delta_step=None,
                                                                         max_depth=5,
                                                                         max_leaves=None,
                                                                         min_child_weight=3,
                                                                         missing=nan,
                                                                         monotone_constraints=None,
                                                                         multi_strategy=None,
                                                                         n_estimators=1000,
                                                                         n_jobs=None,
                                                                         num_parallel_tree=None, ...)),
                       method='isotonic')

In [19]:
# -------------------------------------------------------
# 7. Evaluate
# -------------------------------------------------------
val_preds = calibrated_clf.predict(X_val)
test_preds = calibrated_clf.predict(X_test)

print("\nValidation macro F1:")
print(f1_score(y_val, val_preds, average="macro"))

print("\nTest macro F1:")
print(f1_score(y_test, test_preds, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_preds,
    target_names=label_encoder.classes_
))

print("------------------------------------------------")
val_preds = clf.predict(X_val)
test_preds = clf.predict(X_test)

print("\nValidation macro F1:")
print(f1_score(y_val, val_preds, average="macro"))

print("\nTest macro F1:")
print(f1_score(y_test, test_preds, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_preds,
    target_names=label_encoder.classes_
))


Validation macro F1:
0.8719057567706457

Test macro F1:
0.8638055776692257

Classification Report:
                      precision    recall  f1-score   support

             Anxiety       0.88      0.89      0.89       520
             Bipolar       0.95      0.94      0.94       520
          Depression       0.74      0.70      0.72      1220
              Normal       0.88      0.94      0.91      1220
Personality disorder       0.98      0.96      0.97       420
              Stress       0.92      0.87      0.90       520
            Suicidal       0.72      0.72      0.72      1085

            accuracy                           0.84      5505
           macro avg       0.87      0.86      0.86      5505
        weighted avg       0.83      0.84      0.83      5505

------------------------------------------------

Validation macro F1:
0.8595006036247185

Test macro F1:
0.8584797988949374

Classification Report:
                      precision    recall  f1-score   support

   

In [20]:
# -------------------------------------------------------
# 8. Save model
# -------------------------------------------------------
joblib.dump(clf, "mental_health_xgboost.pkl")
# 3. Save the CALIBRATED model instead of the raw one
joblib.dump(calibrated_clf, 'mental_health_xgboost_CALIBRATED.pkl')

['mental_health_xgboost_CALIBRATED.pkl']

In [32]:
# 1. Load your saved models
embedder = SentenceTransformer('all-MiniLM-L6-v2')
clf = joblib.load('mental_health_xgboost_CALIBRATED.pkl')
label_encoder = joblib.load('label_encoder.pkl')

def get_all_status_probabilities(text):
    """
    Takes raw text, embeds it, and returns the probability 
    for EVERY mental health status.
    """
    # 1. Convert text to embedding
    # We pass it as a list [text] so it returns a 2D array (1 sample, 384 features)
    embedding = embedder.encode([text])
    
    # 2. Get probabilities for ALL classes
    # predict_proba returns an array like [[0.50, 0.20, 0.15, 0.15]]
    # We use [0] to get just the 1D array for our single text
    raw_probabilities = clf.predict_proba(embedding)[0]
    
    # 3. Map the raw numbers to your actual label names
    class_names = label_encoder.classes_
    status_dict = {name: prob for name, prob in zip(class_names, raw_probabilities)}
    
    # 4. Sort the dictionary from highest probability to lowest
    sorted_statuses = dict(sorted(status_dict.items(), key=lambda x: x[1], reverse=True))
    
    return sorted_statuses

# ---------------------------------------------------------
# Test it out!
# ---------------------------------------------------------
user_text = "I love my partner so much, but the second they don't text me back fast enough, I completely split. I get terrified they are abandoning me, so I lash out and push them away before they can leave me. It's an exhausting cycle."
results = get_all_status_probabilities(user_text)

# Print cleanly as percentages
print(f"\nText: '{user_text}'\n")
print("Predicted Probabilities:")
for status, prob in results.items():
    # Format as percentage with 1 decimal place
    print(f" - {status.capitalize():<12}: {prob * 100:5.1f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Text: 'I love my partner so much, but the second they don't text me back fast enough, I completely split. I get terrified they are abandoning me, so I lash out and push them away before they can leave me. It's an exhausting cycle.'

Predicted Probabilities:
 - Personality disorder:  46.5%
 - Suicidal    :  30.1%
 - Depression  :  10.0%
 - Normal      :   6.7%
 - Stress      :   6.1%
 - Bipolar     :   0.4%
 - Anxiety     :   0.2%
